In [1]:
import matplotlib
matplotlib.use('Agg')  # headless rendering
import matplotlib.pyplot as plt
import numpy as np
import neurokit2 as nk
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..') / 'src'))
from ekg_mi.noise import add_noise

NOISE_TYPES = [
    'gaussian',
    'powerline',
    'baseline_wander',
    'muscle_artifact',
    'electrode_motion',
]
SNR_VALUES = [-6, 0, 6, 12, 18, 24]
COLORS = {
    'gaussian':         '#e41a1c',
    'powerline':        '#377eb8',
    'baseline_wander':  '#4daf4a',
    'muscle_artifact':  '#984ea3',
    'electrode_motion': '#ff7f00',
}

FS = 100
ecg = nk.ecg_simulate(duration=10, sampling_rate=FS, heart_rate=70)
t_axis = np.arange(len(ecg)) / FS
freqs = np.fft.rfftfreq(len(ecg), d=1 / FS)
fft_clean = np.abs(np.fft.rfft(ecg))
mask50 = freqs <= 50

print(f'ECG length: {len(ecg)} samples  |  duration: {len(ecg)/FS:.1f} s')

ECG length: 1000 samples  |  duration: 10.0 s


In [2]:
# Pre-warm NSTDB cache so timing doesn't inflate individual cells
from ekg_mi.noise.nstdb import load_nstdb_noise
for t_name in ('baseline_wander', 'muscle_artifact', 'electrode_motion'):
    arr = load_nstdb_noise(t_name)
    print(f'{t_name}: {len(arr):,} samples at 100 Hz')

baseline_wander: 180,556 samples at 100 Hz


muscle_artifact: 180,556 samples at 100 Hz


electrode_motion: 180,556 samples at 100 Hz


In [ ]:
# - Figure 1: Time domain  (5 red × 6 kol) 
fig1, axes1 = plt.subplots(
    len(NOISE_TYPES), len(SNR_VALUES),
    figsize=(18, 12), sharex=True,
)
fig1.suptitle('EKG + Šum - Kroz vrijeme  (sivo = čisto, boja = šum)', fontsize=13)

for row, noise_type in enumerate(NOISE_TYPES):
    color = COLORS[noise_type]
    for col, snr_db in enumerate(SNR_VALUES):
        ax = axes1[row, col]
        noisy = add_noise(ecg, noise_type, snr_db)
        ax.plot(t_axis, ecg,   color='lightgrey', lw=0.6)
        ax.plot(t_axis, noisy, color=color,       lw=0.6, alpha=0.85)
        ax.set_title(f'{snr_db} dB', fontsize=7)
        ax.tick_params(labelsize=6)
        if col == 0:
            ax.set_ylabel(noise_type.replace('_', '\n'), fontsize=7)

plt.tight_layout()
out_dir = Path('../outputs/figures')
out_dir.mkdir(parents=True, exist_ok=True)
fig1.savefig(out_dir / 'noise_time_domain.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 1 saved ', out_dir / 'noise_time_domain.png')

NameError: name 'plt' is not defined

In [ ]:
# - Figure 2: FFT magnituda  (5 redova × 6 kol) 
fig2, axes2 = plt.subplots(
    len(NOISE_TYPES), len(SNR_VALUES),
    figsize=(18, 12), sharex=True,
)
fig2.suptitle(
    'FFT Magnituda (0-50 Hz)  |  sivo = čisto, boja = šum  |  log-scale',
    fontsize=13,
)

eps = 1e-12  # avoid log(0)

for row, noise_type in enumerate(NOISE_TYPES):
    color = COLORS[noise_type]
    for col, snr_db in enumerate(SNR_VALUES):
        ax = axes2[row, col]
        noisy = add_noise(ecg, noise_type, snr_db)
        fft_noisy = np.abs(np.fft.rfft(noisy))
        ax.semilogy(freqs[mask50], fft_clean[mask50] + eps, color='lightgrey', lw=0.8)
        ax.semilogy(freqs[mask50], fft_noisy[mask50] + eps, color=color,       lw=0.8, alpha=0.85)
        ax.set_title(f'{snr_db} dB', fontsize=7)
        ax.tick_params(labelsize=6)
        if col == 0:
            ax.set_ylabel(noise_type.replace('_', '\n'), fontsize=7)
        if row == len(NOISE_TYPES) - 1:
            ax.set_xlabel('Hz', fontsize=7)

plt.tight_layout()
fig2.savefig(out_dir / 'noise_fft.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 2 saved ', out_dir / 'noise_fft.png')

Figure 2 saved → ..\outputs\figures\noise_fft.png


C:\Users\User\AppData\Local\Temp\ipykernel_2372\2175319589.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── SNR error table ────────────────────────────────────────────────────────
print(f'{"noise_type":<22} {"snr_target":>10} {"snr_measured":>13} {"error_dB":>10}')
print('-' * 60)
for noise_type in NOISE_TYPES:
    for snr_db in SNR_VALUES:
        noisy = add_noise(ecg, noise_type, snr_db)
        noise_actual = noisy - ecg
        P_s = np.mean(ecg ** 2)
        P_n = np.mean(noise_actual ** 2)
        snr_meas = 10 * np.log10(P_s / P_n)
        err = snr_meas - snr_db
        flag = ' OK' if abs(err) < 0.5 else ' FAIL'
        print(f'{noise_type:<22} {snr_db:>10} {snr_meas:>13.6f} {err:>+10.2e}{flag}')

noise_type             snr_target  snr_measured   error_dB
------------------------------------------------------------
gaussian                       -6     -6.000000  +0.00e+00 OK
gaussian                        0      0.000000  +9.64e-16 OK
gaussian                        6      6.000000  +8.88e-16 OK
gaussian                       12     12.000000  +0.00e+00 OK
gaussian                       18     18.000000  +0.00e+00 OK
gaussian                       24     24.000000  +0.00e+00 OK
powerline                      -6     -6.000000  +0.00e+00 OK
powerline                       0      0.000000  +0.00e+00 OK
powerline                       6      6.000000  +8.88e-16 OK
powerline                      12     12.000000  +0.00e+00 OK
powerline                      18     18.000000  +0.00e+00 OK
powerline                      24     24.000000  +0.00e+00 OK
baseline_wander                -6     -6.000000  -8.88e-16 OK
baseline_wander                 0      0.000000  +9.64e-16 OK
baseline_wan